# Flow Matching for Discrete Data: Sudoku

Three pretrained 28M-parameter models on *Sudoku Extreme* (9×9 grid, 81-token
context), sharing the same DiT backbone but with different noise processes /
conditioning.

We model discrete sequences $\mathbf{y} = (y^1, \ldots, y^L) \in V^L$ over a
vocabulary $V$ of size $|V|$. Each token $k$ has a learned embedding
$\mathbf{w}_k \in S^{d-1} \subset \mathbb{R}^d$ (a unit vector); these are the
columns of an embedding matrix $W_E \in \mathbb{R}^{d \times |V|}$, normalized
before each use.

1. **vMF** ($d{=}11$). Forward process puts each token's embedding on the unit
   sphere and corrupts via the von Mises–Fisher distribution: the conditional
   path is $p_t(\mathbf{h}\mid \mathbf{w}_k) = f(\mathbf{h};\, \mathbf{w}_k,\, \kappa(t))$ with density
   $f(\mathbf{h};\,\boldsymbol{\mu},\,\kappa) = C_d(\kappa)\,\exp(\kappa\,\boldsymbol{\mu}^\top \mathbf{h})$
   and a monotone schedule $\kappa(t)$ from $\kappa(0)=0$ (uniform on $S^{d-1}$)
   to $\kappa(1)=\kappa_{\max}$.
2. **vMF + time conditioning** (`vmf_tc`). Same noise process; the model also
   receives $\kappa_t / \kappa_{\max}$ via adaLN.
3. **Masked** (MDLM-style baseline). Tokens are masked i.i.d. with probability
   $t$; the reverse CTMC kernel unmasks them progressively.

The notebook is inference-only. Reference: *Spherical Flows for Sampling
Discrete Distributions* ([arXiv:2605.05629](https://arxiv.org/abs/2605.05629)).
The model + sampler code under `flows_categorical/` is copied (with attribution)
from the paper's source repo.


## 0. Setup

Run this once. On Colab it clones the tutorial repo so `flows_categorical/` is
importable, and installs the few extra dependencies. Local users can skip the
clone if they already have the repo.


In [ ]:
# --- Colab setup ---
import os, sys, subprocess

REPO_URL = "https://github.com/JChemseddine/fm_tutorial.git"
REPO_DIR = "fm_tutorial"

ON_COLAB = "google.colab" in sys.modules
if ON_COLAB and not os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
    os.chdir(REPO_DIR)
elif os.path.isdir("flows_categorical"):
    pass  # already at repo root
elif os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)

# Extra deps not in requirements.txt
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "einops", "huggingface_hub", "safetensors"])

# Make `flows_categorical` importable
sys.path.insert(0, os.getcwd())
print("CWD:", os.getcwd())
print("flows_categorical present:", os.path.isdir("flows_categorical"))


In [ ]:
import json
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download, snapshot_download

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Locate pretrained checkpoints

The three checkpoints are ~440 MB each (28M params + Adam state + EMA copy).
Two ways to get them:

- **Local (developer machine)** — drop `final.pt` files into `networks/<method>/`,
  e.g. `networks/vmf_d11_p1/final.pt`. The cell below detects this and skips the
  HF download.
- **Hugging Face Hub** — they live at
  [`Jugc/fm-tutorial`](https://huggingface.co/Jugc/fm-tutorial) with layout

  ```
  Jugc/fm-tutorial/
  ├── vmf_d11_p1/checkpoint.pt
  ├── vmf_tc_d11_p1/checkpoint.pt
  └── masked_p1/checkpoint.pt
  ```

  The notebook auto-falls-back to HF Hub if `networks/` is missing.


In [ ]:
METHOD_NAMES = ["vmf_d11_p1", "vmf_tc_d11_p1", "masked_p1"]
HF_REPO = "Jugc/fm-tutorial"

def resolve_checkpoint(name):
    """Return a local checkpoint path. Prefers ./networks/<name>/final.pt;
    falls back to downloading <name>/checkpoint.pt from HF Hub."""
    local = os.path.join("networks", name, "final.pt")
    if os.path.exists(local):
        return local
    # Download from HF Hub
    return hf_hub_download(repo_id=HF_REPO, filename=f"{name}/checkpoint.pt", repo_type="model")

CKPT_PATHS = {name: resolve_checkpoint(name) for name in METHOD_NAMES}
for name, p in CKPT_PATHS.items():
    src = "local" if p.startswith("networks/") else "HF Hub"
    print(f"  {name:18s}  [{src}]  {p}")


In [ ]:
# Held-out puzzles: bundled in the tutorial repo (100 random puzzles from sudoku-extreme test split)
PUZZLES_PATH = "data/sudoku_extreme_100.npz"
data = np.load(PUZZLES_PATH)
test_inputs = torch.from_numpy(data["inputs"].astype(np.int64))   # (100, 81), 0=blank, 1-9=clue
test_labels = torch.from_numpy(data["labels"].astype(np.int64))   # (100, 81), 1-9 (full solution)

assert test_inputs.min() >= 0 and test_inputs.max() <= 9
assert test_labels.min() >= 1 and test_labels.max() <= 9

print(f"Loaded {len(test_inputs)} test puzzles, shape {tuple(test_inputs.shape)}")
print(f"Avg clues per puzzle: {(test_inputs > 0).float().sum(dim=1).mean():.1f}  "
      f"(min {(test_inputs > 0).sum(dim=1).min().item()}, max {(test_inputs > 0).sum(dim=1).max().item()})")


## 2. Anatomy of a Sudoku puzzle

Each puzzle is a flat sequence of length 81 (the 9×9 grid in row-major order).
Blanks are token 0; the model must predict tokens 1–9 for those positions.
*Sudoku Extreme* puzzles have as few as 17 clues — they are deliberately hard.


In [ ]:
def show_grid(tokens, clue_mask=None, title=None, ax=None):
    """Display a (81,) tensor as a 9x9 grid. Clue cells are bold."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(3.2, 3.2))
    grid = tokens.view(9, 9).cpu().numpy()
    ax.set_xlim(0, 9); ax.set_ylim(9, 0)
    ax.set_xticks([]); ax.set_yticks([])
    for x in range(10):
        lw = 2.0 if x % 3 == 0 else 0.5
        ax.plot([x, x], [0, 9], "k-", lw=lw)
        ax.plot([0, 9], [x, x], "k-", lw=lw)
    cm = None if clue_mask is None else clue_mask.view(9, 9).cpu().numpy()
    for i in range(9):
        for j in range(9):
            v = int(grid[i, j])
            if v == 0:
                continue
            is_clue = cm is not None and bool(cm[i, j])
            ax.text(j + 0.5, i + 0.5, str(v), ha="center", va="center",
                    fontsize=14, fontweight="bold" if is_clue else "normal",
                    color="black" if is_clue else "#1f77b4")
    if title:
        ax.set_title(title, fontsize=10)
    return ax

idx = 0
clue_mask = (test_inputs[idx] != 0)
fig, axes = plt.subplots(1, 2, figsize=(6.4, 3.2))
show_grid(test_inputs[idx], clue_mask, title=f"Puzzle #{idx} (clues bold)", ax=axes[0])
show_grid(test_labels[idx], clue_mask, title="Ground-truth solution", ax=axes[1])
plt.tight_layout(); plt.show()


## 3. Loading the three methods

Each checkpoint ships with the training `config.json` and a `checkpoint.pt`
holding the model weights and (for vMF) the learned $\kappa(t)$ schedule parameters.
A small helper hides the boilerplate.


In [ ]:
from flows_categorical.config import Config
from flows_categorical.model.backbone import ContinuousTransformer, MaskedTransformer
from flows_categorical.methods import create_sampler
from flows_categorical.methods.continuous.spherical.vmf import PsiTable
from flows_categorical.schedule.cdcd_warp import CDCDWarp


def load_method(name, device=DEVICE, use_ema=True):
    """Load (config, model, sampler) for a method name. Uses EMA weights by default."""
    ckpt_path = CKPT_PATHS[name]
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    config = Config.from_dict(ckpt["config"])

    # Backbone
    if config.flow.noise_process == "masked":
        model = MaskedTransformer(config.model.vocab_size, config.model)
    else:
        model = ContinuousTransformer(config.model.vocab_size, config.model)

    # Pick EMA if asked & available; else raw model_state_dict
    src_sd = ckpt.get("ema_state_dict") if use_ema else None
    if src_sd is None:
        src_sd = ckpt["model_state_dict"]
        weight_src = "model_state_dict"
    else:
        weight_src = "ema_state_dict"
    msd = {k.removeprefix("_orig_mod."): v for k, v in src_sd.items()}
    missing, unexpected = model.load_state_dict(msd, strict=False)
    if missing:
        print(f"  [{name}] missing keys: {missing[:3]}{'...' if len(missing) > 3 else ''}")
    if unexpected:
        print(f"  [{name}] unexpected keys: {unexpected[:3]}{'...' if len(unexpected) > 3 else ''}")
    model.to(device).eval()

    # Sampler-side extras
    sampler_kwargs = {}
    if config.flow.noise_process == "vmf":
        sampler_kwargs["psi_table"] = PsiTable(
            config.model.embed_dim,
            kappa_range=config.flow.kappa_max,
            grid_size=config.flow.psi_grid_size,
        )
    if config.flow.use_warp and "warp_state" in ckpt:
        warp = CDCDWarp(
            kappa_max=config.flow.kappa_max,
            num_bins=config.flow.warp_bins,
            warmup_steps=config.flow.time_warp_warmup,
            ema_decay=config.flow.warp_ema_decay,
            noise_increasing=False,  # vMF: parameter (kappa) increases with signal
        )
        warp.load_state_dict(ckpt["warp_state"])
        warp.to(device).eval()
        sampler_kwargs["warp"] = warp

    sampler = create_sampler(model, config, **sampler_kwargs)
    print(f"  [{name}] loaded from {weight_src}  (step {ckpt.get('step','?')})")
    return config, model, sampler


methods = {}
for name in METHOD_NAMES:
    print(f"Loading {name}...")
    methods[name] = load_method(name)

# Quick model-size summary
print()
for name, (cfg, mdl, _) in methods.items():
    n_params = sum(p.numel() for p in mdl.parameters())
    tc = "with t-cond" if cfg.model.time_conditioning else "no t-cond"
    print(f"  {name:18s}  {n_params/1e6:5.1f}M params  noise={cfg.flow.noise_process:6s} d={cfg.model.embed_dim:3d}  {tc}")


## 4. Solving one puzzle with each method

Same puzzle, same clues. The model conditions on the clue positions via
`clue_mask` (which positions are observed) and `clue_values` (the digits at
those positions). The sampler pins the clue embeddings throughout the trajectory.


In [ ]:
def sample_one(sampler, puzzle, num_samples=1, return_intermediates=False):
    """Sample completions for a single puzzle. Returns (tokens, intermediates)."""
    puzzle = puzzle.to(DEVICE)
    clue_mask = (puzzle != 0).unsqueeze(0).expand(num_samples, -1).contiguous()
    clue_values = puzzle.unsqueeze(0).expand(num_samples, -1).contiguous()
    out = sampler.sample(
        num_samples=num_samples, device=DEVICE,
        clue_mask=clue_mask, clue_values=clue_values,
        return_intermediates=return_intermediates, verbose=False,
    )
    return out  # dict with 'tokens', possibly 'intermediates'


idx = 0
puzzle = test_inputs[idx]
clue_mask = (puzzle != 0)
gt = test_labels[idx]

results = {}
for name, (_, _, sampler) in methods.items():
    out = sample_one(sampler, puzzle, num_samples=1)
    results[name] = out["tokens"][0].cpu()

fig, axes = plt.subplots(1, 4, figsize=(12.8, 3.2))
show_grid(puzzle, clue_mask, "Puzzle", axes[0])
for ax, (name, tokens) in zip(axes[1:], results.items()):
    correct = (tokens == gt).float().mean().item() * 100
    show_grid(tokens, clue_mask, f"{name}\n{correct:.0f}% cell-correct", ax=ax)
plt.tight_layout(); plt.show()


## 5. Sampler knobs

Each method exposes a few inference-time choices:

| Knob | What it does | Range |
|------|--------------|-------|
| `sampling_steps` (NFE) | Number of network forward passes during sampling. Higher → smoother trajectory, slower. | 8 – 256 |
| `corrector_steps` | (continuous only) Langevin corrections after each predictor step. 0 disables. | 0 – 8 |
| `corrector_epsilon` | Langevin step size $\varepsilon$. | 0.001 – 0.1 |
| `sampler_method` | `softmax` (single-pass predictor) vs `pc_softmax` (predictor + Langevin corrector). | — |
| `mask_schedule_power` | (masked only) Mask rate $t^p$. $p=1$ linear, $p>1$ unmasks later. | 0.5 – 3 |

The cell below lets you sweep one knob at a time. Default values reproduce the
paper grid at NFE=64.


In [ ]:
# --- Edit me ---
NFE = 64                # sampling steps; 32 fast, 128 careful
CORRECTOR_STEPS = 1     # 0 disables Langevin corrector (only effective with pc_softmax)
CORRECTOR_EPS = 0.01    # Langevin step size
SAMPLER_METHOD = "pc_softmax"  # try "softmax" to see the corrector's effect
# -----------------

def reconfigure(sampler, cfg, **flow_overrides):
    """Mutate a sampler's knobs in place (cheaper than re-creating it)."""
    for k, v in flow_overrides.items():
        if hasattr(sampler, k): setattr(sampler, k, v)
        if hasattr(cfg.flow, k): setattr(cfg.flow, k, v)

# Apply to continuous samplers
for name in ("vmf_d11_p1", "vmf_tc_d11_p1"):
    cfg, _, sampler = methods[name]
    reconfigure(sampler, cfg,
                num_steps=NFE, sampling_steps=NFE,
                corrector_steps=CORRECTOR_STEPS,
                corrector_epsilon=CORRECTOR_EPS,
                method=SAMPLER_METHOD)

# Masked sampler only uses NFE
methods["masked_p1"][2].num_steps = NFE

# Re-sample with new settings
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
show_grid(puzzle, clue_mask, "Puzzle", axes[0])
for ax, name in zip(axes[1:], methods):
    _, _, sampler = methods[name]
    tokens = sample_one(sampler, puzzle, num_samples=1)["tokens"][0].cpu()
    correct = (tokens == gt).float().mean().item() * 100
    show_grid(tokens, clue_mask, f"{name}\nNFE={NFE}  acc={correct:.0f}%", ax=ax)
plt.tight_layout(); plt.show()


## 6. Visualizing the trajectory

Three views of how the model converges on a solution. All three use a single
puzzle and the spherical (vMF) sampler with `return_intermediates=True` so we
record the model state at every step.


In [ ]:
def sample_with_trace(name, idx=0, num_steps=32):
    cfg, model, sampler = methods[name]
    sampler.num_steps = num_steps
    if hasattr(cfg.flow, "sampling_steps"):
        cfg.flow.sampling_steps = num_steps
    puzzle = test_inputs[idx]
    out = sample_one(sampler, puzzle, num_samples=1, return_intermediates=True)
    return out, cfg, model, puzzle


def step_logits(model, h_or_x, name, cfg):
    """Compute (1, 81, V) logits for a snapshot. Works for both continuous and masked."""
    with torch.no_grad():
        if cfg.flow.noise_process == "masked":
            return model(h_or_x.to(DEVICE), clue_mask=None)
        h_prime = model(h_or_x.to(DEVICE), clue_mask=None, sigma=None)
        W_E = model.get_W_E()
        return model.compute_logits(h_prime, W_E=W_E)


# Trace with vMF
out, cfg_vmf, mdl_vmf, puzzle_vmf = sample_with_trace("vmf_d11_p1", idx=0, num_steps=32)
intermediates = out["intermediates"]
print(f"Captured {len(intermediates)} trajectory snapshots from vMF sampler.")


### Viz 1 — argmax digit + confidence at six snapshots in time

For each cell we plot the model's currently-favoured digit, with alpha set to
the max softmax probability. Early on the grid is washed-out (low confidence,
uniform-ish); by the end the model has committed.


In [ ]:
def viz_argmax_strip(intermediates, model, cfg, puzzle, n_panels=6):
    """Six snapshots: argmax digit per cell, alpha = max-prob."""
    clue_mask = (puzzle != 0)
    # Even spacing in trajectory index
    idxs = np.linspace(0, len(intermediates) - 1, n_panels).astype(int)

    fig, axes = plt.subplots(1, n_panels, figsize=(2.2 * n_panels, 2.4))
    for ax, k in zip(axes, idxs):
        snap = intermediates[k]
        h_or_x = snap.get("h_t", snap.get("x_t"))
        logits = step_logits(model, h_or_x, name="", cfg=cfg)  # (1, 81, V)
        probs = F.softmax(logits, dim=-1)[0].cpu()
        confidence, argmax = probs.max(dim=-1)
        # Mask out token 0 (blank) so it doesn't crowd the picture
        argmax = argmax.clamp(min=1)

        ax.set_xlim(0, 9); ax.set_ylim(9, 0); ax.set_xticks([]); ax.set_yticks([])
        for x in range(10):
            lw = 1.5 if x % 3 == 0 else 0.3
            ax.plot([x, x], [0, 9], "k-", lw=lw); ax.plot([0, 9], [x, x], "k-", lw=lw)
        for i in range(9):
            for j in range(9):
                v = int(argmax[i * 9 + j].item())
                a = float(confidence[i * 9 + j].item())
                is_clue = bool(clue_mask[i * 9 + j].item())
                ax.text(j + 0.5, i + 0.5, str(v), ha="center", va="center",
                        fontsize=10, alpha=a if not is_clue else 1.0,
                        color="black" if is_clue else "#1f77b4",
                        fontweight="bold" if is_clue else "normal")
        t = snap.get("time", k / max(1, len(intermediates) - 1))
        ax.set_title(f"t={t:.2f}", fontsize=9)
    plt.tight_layout(); plt.show()

viz_argmax_strip(intermediates, mdl_vmf, cfg_vmf, puzzle_vmf)


### Viz 2 — per-cell entropy heatmap

At each snapshot we compute the entropy of the per-cell predictive distribution
over the 9 digits, $H(p_i) = -\sum_v p_i(v) \log p_i(v)$. High entropy = the model
is uncertain about cell $i$; near 0 = the model has committed. Clue cells stay at 0.


In [ ]:
def viz_entropy_strip(intermediates, model, cfg, puzzle, n_panels=6):
    clue_mask = (puzzle != 0)
    idxs = np.linspace(0, len(intermediates) - 1, n_panels).astype(int)
    fig, axes = plt.subplots(1, n_panels, figsize=(2.2 * n_panels, 2.4))
    log9 = np.log(9.0)  # max entropy over 9 digits
    for ax, k in zip(axes, idxs):
        snap = intermediates[k]
        h_or_x = snap.get("h_t", snap.get("x_t"))
        logits = step_logits(model, h_or_x, name="", cfg=cfg)
        # Restrict to digit tokens 1..9 (drop the 0/blank slot for entropy)
        digit_logits = logits[0, :, 1:10]
        p = F.softmax(digit_logits, dim=-1)
        H = -(p * (p.clamp_min(1e-12).log())).sum(dim=-1).cpu().numpy() / log9  # in [0,1]
        H = H.reshape(9, 9)
        H[clue_mask.view(9, 9).numpy()] = 0.0
        ax.imshow(H, cmap="viridis", vmin=0, vmax=1)
        ax.set_xticks([]); ax.set_yticks([])
        for x in range(10):
            lw = 1.2 if x % 3 == 0 else 0.3
            ax.plot([x - 0.5, x - 0.5], [-0.5, 8.5], "w-", lw=lw)
            ax.plot([-0.5, 8.5], [x - 0.5, x - 0.5], "w-", lw=lw)
        t = snap.get("time", k / max(1, len(intermediates) - 1))
        ax.set_title(f"t={t:.2f}", fontsize=9)
    plt.tight_layout(); plt.show()

viz_entropy_strip(intermediates, mdl_vmf, cfg_vmf, puzzle_vmf)


### Viz 3 — vMF trajectory on the sphere (PCA projection)

For the spherical method, every cell's state $h_t \in S^{d-1}$ traces a curve on
the unit sphere. We project the trajectory of a single non-clue cell to its top
two PCA components (across snapshots), and overlay the 9 digit embeddings as
landmarks. The trajectory should sweep from the uniform centre toward the
embedding of the correct digit.


In [ ]:
def viz_sphere_trajectory(intermediates, model, cfg, puzzle, cell_idx=None):
    if cfg.flow.noise_process != "vmf":
        print("Sphere trajectory viz only applies to vMF — skipping.")
        return
    # Pick a non-clue cell (default: first blank)
    if cell_idx is None:
        non_clue = (puzzle == 0).nonzero().flatten()
        cell_idx = int(non_clue[0].item())
    # Collect (T, d) trajectory for that cell
    traj = torch.stack([snap["h_t"][0, cell_idx] for snap in intermediates], dim=0).cpu()  # (T, d)
    W_E = model.get_W_E().detach().cpu()  # (V, d)
    # Project trajectory + landmarks via PCA on the trajectory points
    centered = traj - traj.mean(0, keepdim=True)
    U, S, Vt = torch.linalg.svd(centered, full_matrices=False)
    PC = Vt[:2]                          # (2, d)
    traj_2d = traj @ PC.T                # (T, 2)
    landmarks_2d = W_E @ PC.T            # (V, 2)

    fig, ax = plt.subplots(figsize=(4.2, 4.2))
    ax.scatter(landmarks_2d[1:10, 0], landmarks_2d[1:10, 1], s=120, c="lightgray",
               edgecolors="black", zorder=2)
    for v in range(1, 10):
        ax.annotate(str(v), (landmarks_2d[v, 0], landmarks_2d[v, 1]),
                    ha="center", va="center", fontsize=10, zorder=3)
    ax.plot(traj_2d[:, 0], traj_2d[:, 1], "-", color="#1f77b4", lw=1.2, zorder=1)
    sc = ax.scatter(traj_2d[:, 0], traj_2d[:, 1], c=range(len(traj_2d)),
                    cmap="plasma", s=18, zorder=2)
    plt.colorbar(sc, ax=ax, label="step"); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"cell {cell_idx} (row {cell_idx // 9}, col {cell_idx % 9})  —  vMF trajectory + digit landmarks", fontsize=9)
    plt.tight_layout(); plt.show()

viz_sphere_trajectory(intermediates, mdl_vmf, cfg_vmf, puzzle_vmf)


## 7. Solving a batch + validity check

For each of `N_PUZZLES` test puzzles we generate one completion with each
method, then check three things:

1. **Cell accuracy** — fraction of non-clue cells matching the ground-truth solution.
2. **Strict-match** — all 81 cells correct.
3. **Sudoku validity** — every row, column, and 3×3 box contains digits 1–9 exactly once.

Validity is a stricter signal than cell accuracy: a method can get 95% of cells
right and still produce an invalid grid.


In [ ]:
def is_valid_sudoku(grid_81):
    g = grid_81.view(9, 9)
    target = torch.arange(1, 10)
    for i in range(9):
        if not torch.equal(torch.sort(g[i]).values, target): return False
        if not torch.equal(torch.sort(g[:, i]).values, target): return False
    for bi in range(3):
        for bj in range(3):
            box = g[3*bi:3*bi+3, 3*bj:3*bj+3].flatten()
            if not torch.equal(torch.sort(box).values, target): return False
    return True


N_PUZZLES = 8  # bump up for a sturdier estimate; ~seconds per puzzle on T4

stats = {name: dict(cell_acc=[], strict=[], valid=[]) for name in methods}
for idx in range(N_PUZZLES):
    puzzle = test_inputs[idx]
    gt = test_labels[idx]
    clue_mask = (puzzle != 0)
    non_clue = ~clue_mask
    for name, (_, _, sampler) in methods.items():
        tokens = sample_one(sampler, puzzle, num_samples=1)["tokens"][0].cpu()
        cell_acc = (tokens[non_clue] == gt[non_clue]).float().mean().item()
        strict = bool((tokens == gt).all())
        valid = is_valid_sudoku(tokens)
        stats[name]["cell_acc"].append(cell_acc)
        stats[name]["strict"].append(strict)
        stats[name]["valid"].append(valid)

print(f"Results over {N_PUZZLES} held-out Sudoku-Extreme puzzles:")
print(f"{'method':<14s}  cell-acc   strict   valid")
for name, d in stats.items():
    print(f"{name:<14s}  {np.mean(d['cell_acc'])*100:6.1f}%   "
          f"{np.mean(d['strict'])*100:5.1f}%   {np.mean(d['valid'])*100:5.1f}%")


## How are these models trained?

This notebook is inference-only, but the relevant equations from the paper:

### vMF / vMF-tc

The conditional probability path on $S^{d-1}$ is
$$p_t(\mathbf{h}\mid \mathbf{w}_k) \;=\; f\!\big(\mathbf{h};\, \mathbf{w}_k,\, \kappa(t)\big)
\;=\; C_d(\kappa(t))\,\exp\!\big(\kappa(t)\,\mathbf{w}_k^\top \mathbf{h}\big),$$
with $C_d(\kappa) = \kappa^{d/2-1} / \big((2\pi)^{d/2}\, I_{d/2-1}(\kappa)\big)$
and $\kappa(t)$ a learned monotone schedule. At $\kappa = 0$ this is uniform on
$S^{d-1}$; as $\kappa \to \infty$ it concentrates on $\mathbf{w}_k$.

For uniform prior $p(k)$ over the vocabulary, the posterior is the softmax
with scale $\kappa$ (Lemma 1 in the paper):
$$p(k \mid \mathbf{h}) \;=\;
\frac{\exp(\kappa\,\mathbf{w}_k^\top \mathbf{h})}{\sum_j \exp(\kappa\,\mathbf{w}_j^\top \mathbf{h})}.$$

The backbone $f_\theta$ maps a noisy state $\mathbf{h}_t \sim p_t(\cdot \mid \mathbf{w}_{y^i})$
to a unit vector $\hat{\mathbf{h}}' \in S^{d-1}$; logits at position $i$ are
$\ell^i_k = \kappa(t)\,\mathbf{w}_k^\top \hat{\mathbf{h}}'^{i}$, and training minimizes the
position-wise cross-entropy
$$\mathcal{L}_{\mathrm{CE}} \;=\; -\,\mathbb{E}_{t, k, \mathbf{h}_t}\!\big[\log p_\theta(k \mid \mathbf{h}_t)\big].$$

The schedule $\kappa(t)$ is itself learned (CDCD-style piecewise-linear warp,
fit to the empirical loss curve). `vmf_tc` additionally feeds $\kappa(t)/\kappa_{\max}$
into the DiT through adaLN; the plain `vmf` model gets no time information.

The drift, Riemannian score, and reverse-SDE drift are all posterior-weighted
tangent sums in $\mathbf{w}_k$ — they reuse the same softmax probabilities computed during
the forward pass (Section 3.4 of the paper). This is how `sampler_method="pc_softmax"`
in this notebook gets both predictor and corrector steps from a single backbone
evaluation per step.

### Masked

Each training step picks $t \sim \mathrm{Unif}(0,1)$ and replaces each token with
a special `[MASK]` symbol independently with probability $t$. The model outputs
logits over $V$ and cross-entropy is computed only on masked positions. At
sampling time, the reverse CTMC kernel unmasks tokens progressively from $t=1$
down to $t \approx 0$; mask-rate schedule is $t^p$ with $p=1$ in our checkpoint
(this is the MDLM line, [Sahoo et al., 2024](https://github.com/kuleshov-group/mdlm)).

Full training code lives in the source repo
[`JChemseddine/spherical`](https://github.com/JChemseddine/spherical) (branch
`paper-release-anon`). Each Sudoku checkpoint took ~1M steps at batch 128.


## Things to try

- **Sweep NFE.** Set `NFE = 8` and watch validity collapse; set `NFE = 256` and see the marginal gain.
- **Toggle the corrector.** Set `SAMPLER_METHOD = "softmax"` vs `"pc_softmax"` — the Langevin corrector trades wall time for fewer constraint violations.
- **Different seeds.** Wrap the sampling cells in `torch.manual_seed(...)`. Two methods disagreeing on a particular cell is informative.
- **Custom puzzle.** Build your own 81-token puzzle (`0` = blank, `1..9` = clue) and feed it through the methods.
- **vmf vs vmf_tc head-to-head.** Run the batch eval a few times and see whether time-conditioning helps on this dataset.

---

The masked-diffusion sampler is based on
[MDLM (Sahoo et al., 2024)](https://github.com/kuleshov-group/mdlm); the DiT
backbone follows Peebles & Xie 2023. See the source repo for full attributions.
